In [1]:
# Librerias

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

In [2]:
# Parametros generales

FASES = ('R', 'S', 'T')

In [62]:
# Parametros configurables PLE1

maq_ple1 = 'PLE1'
maq_ple7 = 'PLE7'
umbral_pico_ple1 = 0.47

k_ple1 = 2
window_ple1 = 3
min_dist_ple1 = 0.01

tol_r_ple1 = 0.045      # tolerancia para determinar si la señal esta dentro de la media en fase R
tol_s_ple1 = 0.067      # tolerancia para determinar si la señal esta dentro de la media en fase S 
tol_t_ple1 = 0.077       # tolerancia para determinar si la señal esta dentro de la media en fase T

t_estable_ple1 = pd.Timedelta(seconds= 27) # Tiempo minimo que tiene que pasar la señal cerca de la media para considerar que el evento termino


In [4]:
# Leer rutas

from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "data"

# print("PROJECT_ROOT:", PROJECT_ROOT)
# print("DATA_PATH:", DATA_PATH)
# print("Existe data:", DATA_PATH.exists())


In [5]:
# Abrir los graficos en el navegador

pio.renderers.default = 'browser'

In [6]:
# Funcion para cargar datos

from pathlib import Path
import pandas as pd

def cargar_maquina(base_path, maquina):
    base_path = Path(base_path).resolve()
    paths = list((base_path / maquina).rglob("*.csv"))

    if not paths:
        raise ValueError(f"No se encontraron CSV para {maquina} en {base_path}")

    dfs = []
    for path in paths:
        df = pd.read_csv(path)
        df["maquina"] = maquina
        dfs.append(df)

    return (
        pd.concat(dfs, ignore_index=True)
          .sort_values("temporal_placa")
          .reset_index(drop=True)
    )


In [7]:
# Carga de datos

df_ple1 = cargar_maquina(DATA_PATH, "PLE1")
df_ple7 = cargar_maquina(DATA_PATH, "PLE7")

In [8]:
# Funcion preparar_df

def preparar_df(df):
    df = df.copy()

    # Timestamp
    df['temporal_placa'] = pd.to_datetime(df['temporal_placa'])
    df['hora'] = df['temporal_placa'].dt.hour
    df['minuto'] = df['temporal_placa'].dt.minute

    # Turnos
    def asignar_turno(hora, minuto):
        t = hora * 60 + minuto

        # Pausas
        if 12*60 <= t < 12*60 + 30:
            return 'ALMUERZO'
        if 22*60 <= t < 22*60 + 30:
            return 'CENA'

        # Turnos
        if 5*60 <= t < 17*60:
            return 'TURNO MAÑANA'
        if 17*60 <= t < 22*60:
            return 'TURNO TARDE'
        if (t >= 22*60 + 30) or (t < 1*60):
            return 'TURNO TARDE'

        return 'FUERA_TURNO'

    df['turno'] = df.apply(
        lambda x: asignar_turno(x['hora'], x['minuto']),
        axis=1
    )
        
    # Potencias totales por timestamp
    df['p_activa_total'] = (
        df['potencia_a_r'] +
        df['potencia_a_s'] +
        df['potencia_a_t']
    )

    df['q_reactiva_total'] = (
        df['potencia_r_r'] +
        df['potencia_r_s'] +
        df['potencia_r_t']
    )

    df['s_aparente'] = np.sqrt(
        df['p_activa_total']**2 +
        df['q_reactiva_total']**2
    )

    return df


In [9]:
# Ajuste de datos de las plegadoras

df_ple1 = preparar_df(df_ple1)
df_ple7 = preparar_df(df_ple7)

df_all = pd.concat([df_ple1, df_ple7], ignore_index=True)

df_all['temporal_placa'] = (
    pd.to_datetime(df_all['temporal_placa'], errors='coerce')
    .dt.tz_localize(None)
)

# Filtra los datos correspondientes al 9-1-2026
import datetime as dt

dia = dt.date(2026, 1, 12)
df_all = df_all[df_all['temporal_placa'].dt.date == dia]

In [10]:
# Calculo la media de las corrientes excluyendo los picos

def media_sin_picos(s, q = 0.9):
    return s[s <= s.quantile(q)].mean()

medias_por_fase = (
    df_all[df_all['maquina'] == maq_ple1]
    .assign(
        R=df_all['corriente_r'],
        S=df_all['corriente_s'],
        T=df_all['corriente_t'],
    )
    .melt(
        id_vars=['maquina'],
        value_vars=['R', 'S', 'T'],
        var_name='fase',
        value_name='corriente'
    )
    .groupby(['maquina', 'fase'], as_index=False)
    .agg(
        baseline=('corriente', media_sin_picos)
    )
)

medias_por_fase

,maquina,fase,baseline
0,PLE1,R,15.800415
1,PLE1,S,16.703100
2,PLE1,T,16.188158


In [63]:
# Funcion detectar_picos mediante corriente

def detectar_picos_i(signal, timestamp, umbral_pico, ventana):

    dt = timestamp.diff().dt.total_seconds()
    valid = dt > 0

    dI_dt = signal.diff() / dt

    # subida fuerte
    subida = valid & dI_dt.notna() & (dI_dt > umbral_pico)

    picos = subida.copy() * False

    # buscar máximo local después de la subida
    for i in range(1, len(signal) - ventana):
        if subida.iloc[i]:
            entorno = signal.iloc[i:i+ventana+1]
            if signal.iloc[i+1] == entorno.max():
                picos.iloc[i+1] = True

    return picos


In [12]:
# Funcion detectar_picos mediante potencia aparente

import numpy as np
import pandas as pd

def detectar_picos_s(
    signal,
    timestamp,
    *,
    k,              # sensibilidad (3–6 típico)
    window,         # vecindad para máximo local
    min_dist_s      # distancia mínima entre picos
):
    s = pd.Series(signal).astype(float).reset_index(drop=True)
    t = pd.to_datetime(timestamp).dt.tz_localize(None).reset_index(drop=True)

    # máximo local (tolera mesetas)
    if window % 2 == 0:
        window += 1
    roll_max = s.rolling(window, center=True, min_periods=1).max()
    is_local_max = (s == roll_max)

    # baseline robusto + ruido
    med = s.rolling(200, center=True, min_periods=20).median()
    resid = s - med
    mad = resid.rolling(200, center=True, min_periods=20).apply(
        lambda x: np.nanmedian(np.abs(x - np.nanmedian(x))),
        raw=True
    )
    sigma = 1.4826 * mad

    thr = med + k * sigma

    candidatos = is_local_max & (s > thr)

    # distancia mínima por tiempo
    idx = np.where(candidatos.fillna(False).values)[0]
    keep = []
    last_t = None

    for i in idx:
        ti = t.iloc[i]
        if last_t is None or (ti - last_t).total_seconds() >= min_dist_s:
            keep.append(i)
            last_t = ti

    mask = np.zeros(len(s), dtype=bool)
    mask[keep] = True

    return pd.Series(mask, index=signal.index)


In [64]:
# Detección de picos en las 3 fases de la PLE1
import pandas as pd

# -----------------------------
# Datos de PLE1
# -----------------------------
df_ple1 = (
    df_all[df_all['maquina'] == maq_ple1]
    .sort_values('temporal_placa')
    .copy()
)

df_ple1['temporal_placa'] = (
    pd.to_datetime(df_ple1['temporal_placa'], errors='coerce')
    .dt.tz_localize(None)
)

# -----------------------------
# Detección de picos en potencia aparente
# -----------------------------
mask_picos_S = detectar_picos_s(
    df_ple1['s_aparente'],
    df_ple1['temporal_placa'],
    k = k_ple1,
    window = window_ple1,
    min_dist_s = min_dist_ple1
)

# -----------------------------
# Deteccion de picos en corriente por fase
# -----------------------------
mask_picos_Ir = detectar_picos_i(
    df_ple1['corriente_r'],
    df_ple1['temporal_placa'],
    umbral_pico=umbral_pico_ple1,
    ventana=5
)

mask_picos_Is = detectar_picos_i(
    df_ple1['corriente_s'],
    df_ple1['temporal_placa'],
    umbral_pico=umbral_pico_ple1,
    ventana=5
)

mask_picos_It = detectar_picos_i(
    df_ple1['corriente_t'],
    df_ple1['temporal_placa'],
    umbral_pico=umbral_pico_ple1,
    ventana=5
)

df_ple1['pico_S']  = mask_picos_S
df_ple1['pico_Ir'] = mask_picos_Ir
df_ple1['pico_Is'] = mask_picos_Is
df_ple1['pico_It'] = mask_picos_It

# -----------------------------
# DataFrame de picos
# -----------------------------
mask_picos_final = (
    df_ple1['pico_S'] |
    df_ple1['pico_Ir'] |
    df_ple1['pico_Is'] |
    df_ple1['pico_It']
)


df_peaks_all_ple1 = df_ple1.loc[
    mask_picos_final,
    ['temporal_placa', 'pico_S', 'pico_Ir', 'pico_Is', 'pico_It']
].copy()

df_peaks_all_ple1['maquina'] = maq_ple1

def origen_pico(row):
    origenes = []
    if row['pico_S']:
        origenes.append('S')
    if row['pico_Ir']:
        origenes.append('Ir')
    if row['pico_Is']:
        origenes.append('Is')
    if row['pico_It']:
        origenes.append('It')
    return '+'.join(origenes)

df_peaks_all_ple1['origen'] = df_peaks_all_ple1.apply(origen_pico, axis=1)


# Ordenar y resetear índice
df_peaks_all_ple1 = (
    df_peaks_all_ple1
    .sort_values('temporal_placa')
    .reset_index(drop=True)
)


In [57]:
# Detección de picos en las 3 fases de la PLE7
import pandas as pd

# -----------------------------
# Datos de PLE1
# -----------------------------
df_ple7 = (
    df_all[df_all['maquina'] == maq_ple7]
    .sort_values('temporal_placa')
    .copy()
)

df_ple7['temporal_placa'] = (
    pd.to_datetime(df_ple7['temporal_placa'], errors='coerce')
    .dt.tz_localize(None)
)

# -----------------------------
# Detección de picos en potencia aparente
# -----------------------------
mask_picos_S = detectar_picos_s(
    df_ple7['s_aparente'],
    df_ple7['temporal_placa'],
    k = k_ple1,
    window = window_ple1,
    min_dist_s = min_dist_ple1
)

# -----------------------------
# Deteccion de picos en corriente por fase
# -----------------------------
mask_picos_Ir = detectar_picos_i(
    df_ple7['corriente_r'],
    df_ple7['temporal_placa'],
    umbral_pico=umbral_pico_ple1,
    ventana=2
)

mask_picos_Is = detectar_picos_i(
    df_ple7['corriente_s'],
    df_ple7['temporal_placa'],
    umbral_pico=umbral_pico_ple1,
    ventana=2
)

mask_picos_It = detectar_picos_i(
    df_ple7['corriente_t'],
    df_ple7['temporal_placa'],
    umbral_pico=umbral_pico_ple1,
    ventana=2
)

# -----------------------------
# DataFrame de picos
# -----------------------------
mask_picos_final = (
    mask_picos_S |
    mask_picos_Ir |
    mask_picos_Is |
    mask_picos_It
)

df_peaks_all_ple7 = df_ple7.loc[
    mask_picos_final,
    ['temporal_placa']
].copy()


df_peaks_all_ple7['maquina'] = maq_ple7
df_peaks_all_ple7['origen'] = 'S'   # pico detectado por potencia

# Ordenar y resetear índice
df_peaks_all_ple7 = (
    df_peaks_all_ple7
    .sort_values('temporal_placa')
    .reset_index(drop=True)
)


In [65]:
# Visualizacion de picos detectados y su marca temporal

df_peaks_all = df_peaks_all_ple1.sort_values(
    by=['maquina', 'temporal_placa']
).reset_index(drop=True)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df_peaks_all_ple1

,temporal_placa,pico_S,pico_Ir,pico_Is,pico_It,maquina,origen
0,2026-01-12 05:00:38,False,False,True,False,PLE1,Is
1,2026-01-12 05:00:41,True,False,False,False,PLE1,S
2,2026-01-12 05:00:43,True,False,False,False,PLE1,S
3,2026-01-12 05:00:46,True,False,False,False,PLE1,S
4,2026-01-12 05:00:51,True,False,False,False,PLE1,S
5,2026-01-12 05:00:56,True,False,False,False,PLE1,S
6,2026-01-12 05:00:58,True,False,False,False,PLE1,S
7,2026-01-12 05:01:01,True,False,False,False,PLE1,S
8,2026-01-12 05:03:02,True,False,False,False,PLE1,S
9,2026-01-12 05:03:07,True,False,False,False,PLE1,S


In [ ]:
# Funcion que detecta eventos

import pandas as pd

def detectar_eventos(
    df_signal,
    df_peaks_fase,
    baseline,
    t_estable,
    maq_obj,
    fase,
    duracion_min_s=5,
    t_fuera_min_s=0.5
):
    """
    Detección de eventos por FSM robusta:
    - tolerancia por fase (% del baseline)
    - histéresis
    - tiempo mínimo fuera de régimen
    - duración mínima de evento
    """

    # -------------------------
    # Tolerancia por fase (%)
    # -------------------------
    if fase == 'R':
        tol = baseline * tol_r_ple1
    elif fase == 'S':
        tol = baseline * tol_s_ple1
    elif fase == 'T':
        tol = baseline * tol_t_ple1
    else:
        raise ValueError(f'Fase desconocida: {fase}')

    # Histéresis
    tol_salida = tol
    tol_retorno = tol * 1.2

    # -------------------------
    # Preparación
    # -------------------------
    picos_set = set(df_peaks_fase['temporal_placa'])

    rows = []

    regimen_confirmado = False
    t_regimen_inicio_global = None

    evento_activo = False
    evento_id = 0
    fecha_inicio = None
    t_regimen_inicio = None
    t_fuera_regimen = None

    # -------------------------
    # Loop principal
    # -------------------------
    for i in range(len(df_signal)):

        t = df_signal.loc[i, 'temporal_placa']
        corriente = df_signal.loc[i, 'corriente']

        hay_pico = t in picos_set

        # Histéresis
        if evento_activo:
            en_regimen = (baseline - tol_retorno <= corriente <= baseline + tol_retorno)
        else:
            en_regimen = (baseline - tol_salida <= corriente <= baseline + tol_salida)

        # -------------------------
        # Warm-up de régimen
        # -------------------------
        if not regimen_confirmado:
            if en_regimen:
                if t_regimen_inicio_global is None:
                    t_regimen_inicio_global = t
                elif t - t_regimen_inicio_global >= t_estable:
                    regimen_confirmado = True
            else:
                t_regimen_inicio_global = None
            continue

        # -------------------------
        # Evento NO activo
        # -------------------------
        if not evento_activo:
            if not en_regimen:
                if t_fuera_regimen is None:
                    t_fuera_regimen = t
                elif t - t_fuera_regimen >= pd.Timedelta(seconds=t_fuera_min_s):
                    if hay_pico:
                        evento_activo = True
                        evento_id += 1
                        fecha_inicio = t_fuera_regimen
                        t_regimen_inicio = None
                        t_fuera_regimen = None
            else:
                t_fuera_regimen = None
            continue

        # -------------------------
        # Evento activo
        # -------------------------
        if not en_regimen or hay_pico:
            t_regimen_inicio = None
            continue

        if t_regimen_inicio is None:
            t_regimen_inicio = t
        elif t - t_regimen_inicio >= t_estable:

            duracion_s = (t_regimen_inicio - fecha_inicio).total_seconds()

            if duracion_s >= duracion_min_s:
                rows.append({
                    'maquina': maq_obj,
                    'fase': fase,
                    'ID_Evento': evento_id,
                    'Fecha Inicio': fecha_inicio,
                    'Fecha Fin': t_regimen_inicio,
                    'Duracion Evento [s]': duracion_s
                })

            evento_activo = False
            fecha_inicio = None
            t_regimen_inicio = None

    # -------------------------
    # Evento abierto al final
    # -------------------------
    if evento_activo and fecha_inicio is not None:
        t_fin = df_signal.iloc[-1]['temporal_placa']
        duracion_s = (t_fin - fecha_inicio).total_seconds()

        if duracion_s >= duracion_min_s:
            rows.append({
                'maquina': maq_obj,
                'fase': fase,
                'ID_Evento': evento_id,
                'Fecha Inicio': fecha_inicio,
                'Fecha Fin': t_fin,
                'Duracion Evento [s]': duracion_s
            })

    return rows


In [73]:
# Analisis de eventos

import pandas as pd

# =================================================
# 1) PREPROCESAR SEÑALES
# =================================================

signal_dict = {}

df_all_ple1 = df_all[df_all['maquina'] == maq_ple1].copy()

df_all_ple1['temporal_placa'] = (
    pd.to_datetime(df_all_ple1['temporal_placa'])
    .dt.tz_localize(None)
)

for fase in FASES:
    col = f'corriente_{fase.lower()}'
    signal_dict[fase] = (
        df_all_ple1[['temporal_placa', col]]
        .rename(columns={col: 'corriente'})
        .sort_values('temporal_placa')
        .reset_index(drop=True)
    )

# =================================================
# 2) BASELINE POR FASE (ROBUSTO)
# =================================================

baseline_fase = (
    medias_por_fase
    .set_index('fase')['baseline']
    .to_dict()
)

# =================================================
# 3) PICOS DE LA MAQUINA
# =================================================

df_peaks = df_peaks_all_ple1[
    df_peaks_all_ple1['maquina'] == maq_ple1
].copy()

df_peaks['temporal_placa'] = (
    pd.to_datetime(df_peaks['temporal_placa'])
    .dt.tz_localize(None)
)

rows = []

for fase in FASES:

    fase = fase.upper().strip()
    if fase not in baseline_fase:
        continue

    df_signal = signal_dict[fase]

    #df_peaks_fase = (
        #df_peaks[df_peaks['fase'].str.upper() == fase]
        #.sort_values('temporal_placa')
        #.reset_index(drop=True)
    #)

    if df_peaks.empty:
        continue

    rows.extend(
        detectar_eventos(
            df_signal=df_signal,
            df_peaks_fase=df_peaks,
            baseline=baseline_fase[fase],
            t_estable=t_estable_ple1,
            maq_obj=maq_ple1,
            fase=fase
        )
    )

# =================================================
# 5) DATAFRAME FINAL
# =================================================

df_evento_ple1 = (
    pd.DataFrame(rows)
    .sort_values(by=['ID_Evento'])
    .reset_index(drop=True)
)

df_evento_ple1['Fecha Inicio'] = pd.to_datetime(df_evento_ple1['Fecha Inicio'])
df_evento_ple1['Fecha Fin'] = pd.to_datetime(df_evento_ple1['Fecha Fin'])

df_evento_ple1['duracion_s'] = (
    df_evento_ple1['Fecha Fin'] - df_evento_ple1['Fecha Inicio']
).dt.total_seconds()

df_evento_ple1 = df_evento_ple1[df_evento_ple1['duracion_s'] >= 3].reset_index(drop=True)

df_evento_ple1

,maquina,fase,ID_Evento,Fecha Inicio,Fecha Fin,Duracion Evento [s],Picos,duracion_s
0,PLE1,R,1,2026-01-12 05:20:18,2026-01-12 05:20:56,38.0,18,38.0
1,PLE1,S,1,2026-01-12 05:20:18,2026-01-12 05:20:54,36.0,18,36.0
2,PLE1,T,1,2026-01-12 05:20:18,2026-01-12 05:20:54,36.0,18,36.0
3,PLE1,S,2,2026-01-12 05:23:30,2026-01-12 05:24:11,41.0,20,41.0
4,PLE1,T,2,2026-01-12 05:23:30,2026-01-12 05:24:06,36.0,20,36.0
5,PLE1,R,2,2026-01-12 05:23:30,2026-01-12 05:24:11,41.0,20,41.0
6,PLE1,R,3,2026-01-12 05:26:55,2026-01-12 05:27:23,28.0,16,28.0
7,PLE1,S,3,2026-01-12 05:26:55,2026-01-12 05:27:23,28.0,16,28.0
8,PLE1,T,3,2026-01-12 05:26:55,2026-01-12 05:27:23,28.0,16,28.0
9,PLE1,T,4,2026-01-12 05:30:55,2026-01-12 05:32:15,80.0,42,80.0


In [74]:
# Grafica de eventos para PLE1

import plotly.graph_objects as go

maq = maq_ple1  # solo PLE1

map_fase_col = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

colores_fase = {
    'R': 'red',
    'S': 'green',
    'T': 'blue'
}

fig = go.Figure()

df_m = (
    df_all[df_all['maquina'] == maq]
    .sort_values('temporal_placa')
)

# Trazas de corriente
trace_idx = {}

for fase, col in map_fase_col.items():

    visible = (fase == 'R')  # fase inicial

    fig.add_trace(
        go.Scatter(
            x=df_m['temporal_placa'],
            y=df_m[col],
            mode='lines',
            name=f'Fase {fase}',
            line=dict(color=colores_fase[fase]),
            visible=visible
        )
    )

    trace_idx[fase] = len(fig.data) - 1

# Shapes por fase
shapes_por_fase = {}

for fase in map_fase_col.keys():

    shapes = []

    df_e = df_evento_ple1[
        (df_evento_ple1['maquina'] == maq) &
        (df_evento_ple1['fase'] == fase)
    ]

    for _, ev in df_e.iterrows():
        shapes.append(
            dict(
                type='rect',
                xref='x',
                yref='paper',
                x0=ev['Fecha Inicio'],
                x1=ev['Fecha Fin'],
                y0=0,
                y1=1,
                fillcolor=colores_fase[fase],
                opacity=0.12,
                line_width=0
            )
        )

    shapes_por_fase[fase] = shapes

botones = []

for fase in map_fase_col.keys():

    visibles = [False] * len(fig.data)
    visibles[trace_idx[fase]] = True

    botones.append(
        dict(
            label=f'Fase {fase}',
            method='update',
            args=[
                {'visible': visibles},
                {'shapes': shapes_por_fase[fase]}
            ]
        )
    )

# Layout
fig.update_layout(
    title='PLE1 – Corriente por fase con eventos detectados',
    xaxis_title='Tiempo',
    yaxis_title='Corriente [A]',
    template='plotly_white',
    shapes=shapes_por_fase['R'],  # fase inicial
    updatemenus=[
        dict(
            buttons=botones,
            direction='down',
            x=1.08,
            y=1.1,
            showactive=True
        )
    ]
)

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'Umbral pico: {umbral_pico_ple1}',
    line=dict(color='black'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'K: {k_ple1}',
    line=dict(color='gray'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'window: {window_ple1}',
    line=dict(color='gray'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'min_dist_s: {min_dist_ple1}',
    line=dict(color='gray'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'tolerancia R: {tol_r_ple1}',
    line=dict(color='gray'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'tolerancia S: {tol_s_ple1}',
    line=dict(color='gray'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    name=f'tolerancia T: {tol_t_ple1}',
    line=dict(color='gray'),
    showlegend=True
))

fig.show()



In [60]:
# Grafica de eventos por fases superpuestas para PLE1

import plotly.graph_objects as go

maq = maq_ple1

map_fase_col = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

colores_fase = {
    'R': 'red',
    'S': 'green',
    'T': 'blue'
}

fig = go.Figure()

df_m = (
    df_all[df_all['maquina'] == maq]
    .sort_values('temporal_placa')
)

# Corrientes (3 fases superpuestas)
for fase, col in map_fase_col.items():
    fig.add_trace(
        go.Scatter(
            x=df_m['temporal_placa'],
            y=df_m[col],
            mode='lines',
            name=f'Fase {fase}',
            line=dict(color=colores_fase[fase])
        )
    )

# Shapes: eventos por fase (PLE1)
shapes = []

for fase in map_fase_col.keys():

    df_e = df_evento_ple1[
        (df_evento_ple1['maquina'] == maq) &
        (df_evento_ple1['fase'] == fase)
    ]

    for _, ev in df_e.iterrows():
        shapes.append(
            dict(
                type='rect',
                xref='x',
                yref='paper',
                x0=ev['Fecha Inicio'],
                x1=ev['Fecha Fin'],
                y0=0,
                y1=1,
                fillcolor=colores_fase[fase],
                opacity=0.12,   # claro, no tapa la señal
                line_width=0
            )
        )

# Layout final
fig.update_layout(
    title='PLE1 – Corriente por fase con eventos detectados',
    xaxis_title='Tiempo',
    yaxis_title='Corriente [A]',
    shapes=shapes,
    legend_title='Fases',
    template='plotly_white'
)

fig.show()


In [ ]:
# Eventos a .cvs
from pathlib import Path

OUT_TABLAS = PROJECT_ROOT / "output" / "tablas"
OUT_TABLAS.mkdir(parents=True, exist_ok=True)

df_evento_ple1.to_csv(OUT_TABLAS / "df_eventos_ple1.csv", index=False)